In [1]:
# ============================================================
# CELL 0: CLEAR CORRUPTED CACHE
# ============================================================
import os
import shutil

print("🧹 Clearing corrupted Python cache...")

# Clear __pycache__ in site-packages
site_packages = None
for path in os.listdir(r"d:\projects\automation project\.venv\Lib\site-packages"):
    if path == "chromadb":
        site_packages = r"d:\projects\automation project\.venv\Lib\site-packages"
        break

if site_packages:
    count = 0
    for root, dirs, files in os.walk(site_packages):
        if '__pycache__' in dirs:
            pycache = os.path.join(root, '__pycache__')
            try:
                shutil.rmtree(pycache)
                count += 1
            except:
                pass
    print(f"✅ Cleared {count} cache folders")
else:
    print("⚠️  Could not find site-packages")

print("🔴 NOW RESTART THE KERNEL (Kernel → Restart)")
print("   Then run the rest of the notebook")

🧹 Clearing corrupted Python cache...
⚠️  Could not find site-packages
🔴 NOW RESTART THE KERNEL (Kernel → Restart)
   Then run the rest of the notebook


In [2]:
"""
build_kb.py - Knowledge Base Builder
Creates vector embeddings for 100 products across 5 categories
Stores in ChromaDB with cosine similarity for semantic search

Usage: python build_kb.py
"""

import json
import os
import sys
from pathlib import Path
from typing import Dict, List, Any, Optional
import chromadb
from sentence_transformers import SentenceTransformer
import torch
from tqdm import tqdm
import hashlib

C:\Users\pmthe\product-gpt\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ============================================================================
# CONFIGURATION (Easy to change later)
# ============================================================================

MODEL_NAME = os.getenv("EMBEDDING_MODEL", "all-MiniLM-L6-v2")
DB_PATH = os.getenv("CHROMA_DB_PATH", "database/chroma_db")
COLLECTION_NAME = "products"
BATCH_SIZE = 100
SIMILARITY_METRIC = "cosine"

In [4]:
# ============================================================================
# PRODUCT DATA - 100 PRODUCTS ACROSS 5 CATEGORIES
# ============================================================================

LAPTOPS_DATA = [
    {"brand": "Lenovo", "model": "ThinkPad E15", "processor": "Intel i5-12450H", "ram": "8GB", "storage": "256GB SSD", "display": "15.6 FHD", "graphics": "Integrated", "price": 45000, "category": "business", "battery": 52, "weight": 1.8},
    {"brand": "Dell", "model": "Inspiron 15 3000", "processor": "Intel i3-1215U", "ram": "8GB", "storage": "256GB SSD", "display": "15.6 FHD", "graphics": "Integrated", "price": 35000, "category": "student", "battery": 42, "weight": 1.9},
    {"brand": "HP", "model": "Pavilion 15", "processor": "AMD Ryzen 5 5500U", "ram": "8GB", "storage": "512GB SSD", "display": "15.6 FHD", "graphics": "Integrated", "price": 42000, "category": "student", "battery": 41, "weight": 1.75},
    {"brand": "ASUS", "model": "VivoBook 14", "processor": "Intel i5-1235U", "ram": "8GB", "storage": "512GB SSD", "display": "14 FHD", "graphics": "Integrated", "price": 48000, "category": "ultrabook", "battery": 50, "weight": 1.4},
    {"brand": "Acer", "model": "Aspire 5", "processor": "AMD Ryzen 5 5600H", "ram": "16GB", "storage": "512GB SSD", "display": "15.6 FHD", "graphics": "RTX 3050", "price": 65000, "category": "gaming", "battery": 48, "weight": 1.9},
    {"brand": "MSI", "model": "GF63 Thin", "processor": "Intel i5-11400H", "ram": "16GB", "storage": "512GB SSD", "display": "15.6 FHD", "graphics": "RTX 3050 Ti", "price": 72000, "category": "gaming", "battery": 45, "weight": 1.86},
    {"brand": "Lenovo", "model": "IdeaPad 5 Pro", "processor": "AMD Ryzen 7 5800H", "ram": "16GB", "storage": "512GB SSD", "display": "16 WUXGA+", "graphics": "RTX 3050", "price": 80000, "category": "content-creation", "battery": 52, "weight": 1.92},
    {"brand": "Dell", "model": "XPS 13", "processor": "Intel i7-1280P", "ram": "16GB", "storage": "512GB SSD", "display": "13.3 FHD+", "graphics": "Integrated", "price": 115000, "category": "ultrabook", "battery": 52, "weight": 1.2},
    {"brand": "HP", "model": "Envy 13", "processor": "Intel i5-1240P", "ram": "8GB", "storage": "512GB SSD", "display": "13.3 FHD", "graphics": "Integrated", "price": 65000, "category": "ultrabook", "battery": 52, "weight": 1.3},
    {"brand": "ASUS", "model": "TUF Gaming F15", "processor": "Intel i7-12700H", "ram": "16GB", "storage": "512GB SSD", "display": "15.6 FHD 165Hz", "graphics": "RTX 3060", "price": 95000, "category": "gaming", "battery": 90, "weight": 2.1},
    {"brand": "Acer", "model": "Swift 3", "processor": "Intel i5-1240P", "ram": "8GB", "storage": "256GB SSD", "display": "14 FHD", "graphics": "Integrated", "price": 52000, "category": "student", "battery": 54, "weight": 1.35},
    {"brand": "MSI", "model": "Creator 15", "processor": "Intel i9-12900HK", "ram": "32GB", "storage": "1TB SSD", "display": "15.6 4K OLED", "graphics": "RTX 3080 Ti", "price": 180000, "category": "content-creation", "battery": 60, "weight": 2.5},
    {"brand": "Lenovo", "model": "Yoga 9i", "processor": "Intel i7-12700H", "ram": "16GB", "storage": "512GB SSD", "display": "14 OLED Touch", "graphics": "RTX 3050 Ti", "price": 125000, "category": "content-creation", "battery": 60, "weight": 1.6},
    {"brand": "Dell", "model": "Alienware m15 R7", "processor": "Intel i9-12900HK", "ram": "32GB", "storage": "1TB SSD", "display": "15.6 FHD 360Hz", "graphics": "RTX 3080 Ti", "price": 220000, "category": "gaming", "battery": 80, "weight": 2.6},
    {"brand": "HP", "model": "Spectre x360", "processor": "Intel i7-1280P", "ram": "16GB", "storage": "512GB SSD", "display": "13.5 OLED Touch", "graphics": "Integrated", "price": 135000, "category": "ultrabook", "battery": 61, "weight": 1.34},
    {"brand": "ASUS", "model": "ROG Zephyrus G14", "processor": "AMD Ryzen 9 6900HX", "ram": "16GB", "storage": "1TB SSD", "display": "14 QHD 144Hz", "graphics": "RTX 3060", "price": 140000, "category": "gaming", "battery": 80, "weight": 1.65},
    {"brand": "Acer", "model": "Predator Triton 300", "processor": "Intel i7-12700H", "ram": "16GB", "storage": "512GB SSD", "display": "15.6 QHD 165Hz", "graphics": "RTX 3080 Ti", "price": 160000, "category": "gaming", "battery": 100, "weight": 2.15},
    {"brand": "Lenovo", "model": "ThinkBook 14", "processor": "Intel i5-1240P", "ram": "8GB", "storage": "256GB SSD", "display": "14 FHD IPS", "graphics": "Integrated", "price": 48000, "category": "business", "battery": 52, "weight": 1.4},
    {"brand": "Dell", "model": "Inspiron 16 Plus", "processor": "Intel i7-12700H", "ram": "16GB", "storage": "512GB SSD", "display": "16 QHD+", "graphics": "RTX 3050 Ti", "price": 105000, "category": "content-creation", "battery": 60, "weight": 2.1},
    {"brand": "HP", "model": "ZBook Firefly", "processor": "Intel i5-12500H", "ram": "16GB", "storage": "512GB SSD", "display": "14 FHD", "graphics": "Integrated", "price": 70000, "category": "business", "battery": 55, "weight": 1.42},
    {"brand": "ASUS", "model": "ProBook 450", "processor": "Intel i5-1240P", "ram": "8GB", "storage": "256GB SSD", "display": "15.6 FHD", "graphics": "Integrated", "price": 45000, "category": "business", "battery": 52, "weight": 1.8},
    {"brand": "Acer", "model": "Chromebook Spin 11", "processor": "Intel Celeron N3350", "ram": "4GB", "storage": "64GB eMMC", "display": "11.6 HD Touch", "graphics": "Integrated", "price": 25000, "category": "student", "battery": 48, "weight": 1.2},
    {"brand": "Lenovo", "model": "Legion 5 Pro", "processor": "Intel i7-12700H", "ram": "16GB", "storage": "512GB SSD", "display": "16 QHD 165Hz", "graphics": "RTX 3070 Ti", "price": 145000, "category": "gaming", "battery": 80, "weight": 2.5},
    {"brand": "Dell", "model": "Precision 5560", "processor": "Intel i7-12700H", "ram": "32GB", "storage": "1TB SSD", "display": "15.6 OLED 4K", "graphics": "RTX A2000", "price": 175000, "category": "content-creation", "battery": 56, "weight": 2.0},
    {"brand": "HP", "model": "Omen 15", "processor": "Intel i9-12900HK", "ram": "32GB", "storage": "1TB SSD", "display": "15.6 FHD 165Hz", "graphics": "RTX 3080 Ti", "price": 185000, "category": "gaming", "battery": 70, "weight": 2.45},
    {"brand": "ASUS", "model": "StudioBook 16", "processor": "Intel i7-12700H", "ram": "32GB", "storage": "1TB SSD", "display": "16 OLED 4K", "graphics": "RTX A1000", "price": 200000, "category": "content-creation", "battery": 63, "weight": 2.2},
    {"brand": "Acer", "model": "ConceptD 7", "processor": "Intel i9-12900HK", "ram": "32GB", "storage": "1TB SSD", "display": "15.6 OLED 4K", "graphics": "RTX 3080 Ti", "price": 210000, "category": "content-creation", "battery": 63, "weight": 2.3},
    {"brand": "Lenovo", "model": "ThinkPad P1", "processor": "Intel i9-12900HK", "ram": "32GB", "storage": "1TB SSD", "display": "16 OLED 4K", "graphics": "RTX A5500", "price": 240000, "category": "content-creation", "battery": 63, "weight": 1.99},
    {"brand": "Dell", "model": "Alienware x17 R2", "processor": "Intel i9-12900HK", "ram": "32GB", "storage": "2TB SSD", "display": "17.3 QHD 360Hz", "graphics": "RTX 3080 Ti", "price": 250000, "category": "gaming", "battery": 86, "weight": 3.0},
    {"brand": "MSI", "model": "Stealth 17 Studio", "processor": "Intel i9-12900HK", "ram": "32GB", "storage": "2TB SSD", "display": "17.3 OLED 4K", "graphics": "RTX 3080 Ti", "price": 280000, "category": "content-creation", "battery": 62, "weight": 2.6},
]

PHONES_DATA = [
    {"brand": "Samsung", "model": "Galaxy A13", "processor": "MediaTek Helio G80", "ram": "4GB", "storage": "64GB", "camera": "50MP", "display": "6.6 IPS FHD+", "battery": 5000, "os": "Android 12", "price": 14000, "category": "budget"},
    {"brand": "Xiaomi", "model": "Redmi Note 11", "processor": "MediaTek Helio G99", "ram": "4GB", "storage": "64GB", "camera": "50MP", "display": "6.43 AMOLED FHD+", "battery": 5000, "os": "Android 12", "price": 16000, "category": "budget"},
    {"brand": "OnePlus", "model": "Nord CE 2", "processor": "MediaTek Dimensity 900", "ram": "6GB", "storage": "128GB", "camera": "64MP", "display": "6.43 AMOLED FHD+", "battery": 4500, "os": "Android 12", "price": 24000, "category": "mid-range"},
    {"brand": "Google", "model": "Pixel 6a", "processor": "Google Tensor", "ram": "6GB", "storage": "128GB", "camera": "12.2MP Dual", "display": "6.1 OLED FHD", "battery": 4410, "os": "Android 13", "price": 28000, "category": "mid-range"},
    {"brand": "Samsung", "model": "Galaxy A32", "processor": "MediaTek Helio G80", "ram": "4GB", "storage": "64GB", "camera": "48MP", "display": "6.4 AMOLED FHD", "battery": 5000, "os": "Android 11", "price": 18000, "category": "budget"},
    {"brand": "Xiaomi", "model": "Poco X4", "processor": "MediaTek Helio G96", "ram": "6GB", "storage": "128GB", "camera": "50MP", "display": "6.67 AMOLED FHD+ 90Hz", "battery": 5000, "os": "Android 11", "price": 18500, "category": "mid-range"},
    {"brand": "OnePlus", "model": "9", "processor": "Snapdragon 888", "ram": "8GB", "storage": "128GB", "camera": "48MP Triple", "display": "6.55 AMOLED FHD+ 120Hz", "battery": 4500, "os": "Android 12", "price": 42000, "category": "flagship"},
    {"brand": "Samsung", "model": "Galaxy S22", "processor": "Snapdragon 8 Gen 1", "ram": "8GB", "storage": "128GB", "camera": "50MP Triple", "display": "6.1 AMOLED QHD 120Hz", "battery": 3700, "os": "Android 12", "price": 72000, "category": "flagship"},
    {"brand": "Apple", "model": "iPhone 13", "processor": "Apple A15 Bionic", "ram": "4GB", "storage": "128GB", "camera": "12MP Dual", "display": "6.1 Super Retina XDR", "battery": 3240, "os": "iOS 16", "price": 79000, "category": "flagship"},
    {"brand": "Nothing", "model": "Phone 1", "processor": "Snapdragon 778+", "ram": "8GB", "storage": "128GB", "camera": "50MP Dual", "display": "6.55 OLED FHD+ 120Hz", "battery": 4500, "os": "Android 12", "price": 32000, "category": "mid-range"},
    {"brand": "Google", "model": "Pixel 7", "processor": "Google Tensor", "ram": "8GB", "storage": "128GB", "camera": "50MP Dual", "display": "6.3 OLED FHD 90Hz", "battery": 4355, "os": "Android 13", "price": 59000, "category": "flagship"},
    {"brand": "Xiaomi", "model": "12 Pro", "processor": "Snapdragon 8 Gen 1", "ram": "12GB", "storage": "256GB", "camera": "50MP Triple", "display": "6.73 AMOLED QHD+ 120Hz", "battery": 4600, "os": "Android 12", "price": 72000, "category": "flagship"},
    {"brand": "OnePlus", "model": "10 Pro", "processor": "Snapdragon 8 Gen 1", "ram": "12GB", "storage": "256GB", "camera": "48MP Triple", "display": "6.7 AMOLED QHD 120Hz", "battery": 5000, "os": "Android 12", "price": 71000, "category": "flagship"},
    {"brand": "Samsung", "model": "Galaxy A52", "processor": "Snapdragon 720G", "ram": "6GB", "storage": "128GB", "camera": "64MP Quad", "display": "6.5 AMOLED FHD+ 90Hz", "battery": 4500, "os": "Android 12", "price": 28000, "category": "mid-range"},
    {"brand": "Xiaomi", "model": "Redmi Note 10", "processor": "Snapdragon 678", "ram": "4GB", "storage": "64GB", "camera": "48MP", "display": "6.43 AMOLED FHD+ 60Hz", "battery": 5000, "os": "Android 11", "price": 15000, "category": "budget"},
    {"brand": "Apple", "model": "iPhone 12", "processor": "Apple A14 Bionic", "ram": "4GB", "storage": "64GB", "camera": "12MP Dual", "display": "6.1 Super Retina XDR", "battery": 2815, "os": "iOS 15", "price": 65000, "category": "flagship"},
    {"brand": "OnePlus", "model": "10T", "processor": "Snapdragon 8+ Gen 1", "ram": "12GB", "storage": "256GB", "camera": "50MP Triple", "display": "6.7 AMOLED FHD 120Hz", "battery": 4700, "os": "Android 12", "price": 49000, "category": "flagship"},
    {"brand": "Google", "model": "Pixel 6 Pro", "processor": "Google Tensor", "ram": "12GB", "storage": "256GB", "camera": "50MP Triple", "display": "6.7 OLED QHD 120Hz", "battery": 5003, "os": "Android 13", "price": 71000, "category": "flagship"},
    {"brand": "Samsung", "model": "Galaxy M32", "processor": "MediaTek Helio G80", "ram": "4GB", "storage": "64GB", "camera": "48MP", "display": "6.4 IPS FHD", "battery": 5000, "os": "Android 12", "price": 15000, "category": "budget"},
    {"brand": "Xiaomi", "model": "11i HyperCharge", "processor": "MediaTek Dimensity 920", "ram": "8GB", "storage": "128GB", "camera": "108MP", "display": "6.67 IPS FHD+ 120Hz", "battery": 4500, "os": "Android 12", "price": 26000, "category": "mid-range"},
    {"brand": "Nothing", "model": "Phone 1a", "processor": "Snapdragon 778G+", "ram": "6GB", "storage": "128GB", "camera": "50MP Dual", "display": "6.55 OLED FHD+ 90Hz", "battery": 4500, "os": "Android 13", "price": 24000, "category": "mid-range"},
    {"brand": "Apple", "model": "iPhone 14", "processor": "Apple A16 Bionic", "ram": "6GB", "storage": "128GB", "camera": "12MP Dual", "display": "6.1 Super Retina XDR", "battery": 3279, "os": "iOS 16", "price": 79000, "category": "flagship"},
    {"brand": "Samsung", "model": "Galaxy S21", "processor": "Snapdragon 888", "ram": "8GB", "storage": "128GB", "camera": "64MP Triple", "display": "6.2 AMOLED FHD+ 120Hz", "battery": 4000, "os": "Android 12", "price": 61000, "category": "flagship"},
    {"brand": "OnePlus", "model": "11", "processor": "Snapdragon 8 Gen 1", "ram": "8GB", "storage": "128GB", "camera": "50MP Triple", "display": "6.7 AMOLED FHD+ 120Hz", "battery": 5000, "os": "Android 13", "price": 55000, "category": "flagship"},
    {"brand": "Xiaomi", "model": "Poco F4", "processor": "Snapdragon 870", "ram": "6GB", "storage": "128GB", "camera": "64MP", "display": "6.67 AMOLED FHD+ 120Hz", "battery": 4500, "os": "Android 12", "price": 27000, "category": "mid-range"},
]

TABLETS_DATA = [
    {"brand": "Apple", "model": "iPad (10th Gen)", "display": "10.9 Liquid Retina", "processor": "Apple A14 Bionic", "ram": "4GB", "storage": "64GB", "battery": 7606, "stylus_support": False, "price": 35000, "category": "entertainment"},
    {"brand": "Samsung", "model": "Galaxy Tab A7", "display": "10.4 IPS FHD", "processor": "MediaTek Helio G80", "ram": "3GB", "storage": "32GB", "battery": 7040, "stylus_support": False, "price": 16000, "category": "entertainment"},
    {"brand": "Lenovo", "model": "Tab P11 Pro", "display": "11.5 OLED QHD+", "processor": "MediaTek Kompanio 1300T", "ram": "8GB", "storage": "128GB", "battery": 8200, "stylus_support": True, "price": 55000, "category": "professional"},
    {"brand": "Xiaomi", "model": "Pad 5", "display": "11 AMOLED QHD+", "processor": "Snapdragon 870", "ram": "6GB", "storage": "128GB", "battery": 8720, "stylus_support": True, "price": 32000, "category": "professional"},
    {"brand": "Apple", "model": "iPad Air (5th Gen)", "display": "10.9 Liquid Retina", "processor": "Apple M1", "ram": "8GB", "storage": "64GB", "battery": 7606, "stylus_support": True, "price": 58000, "category": "professional"},
    {"brand": "Samsung", "model": "Galaxy Tab S8", "display": "11 AMOLED QHD 120Hz", "processor": "Snapdragon 8 Gen 1", "ram": "8GB", "storage": "128GB", "battery": 8000, "stylus_support": True, "price": 56000, "category": "professional"},
    {"brand": "OnePlus", "model": "Pad", "display": "11.61 AMOLED QHD 144Hz", "processor": "MediaTek Dimensity 9000", "ram": "8GB", "storage": "128GB", "battery": 9510, "stylus_support": False, "price": 38000, "category": "entertainment"},
    {"brand": "Lenovo", "model": "Tab M10 Plus (3rd Gen)", "display": "10.3 IPS FHD", "processor": "MediaTek Helio G80", "ram": "4GB", "storage": "64GB", "battery": 7700, "stylus_support": False, "price": 18000, "category": "entertainment"},
    {"brand": "Apple", "model": "iPad Pro 11 (4th Gen)", "display": "11 Liquid Retina XDR", "processor": "Apple M2", "ram": "8GB", "storage": "128GB", "battery": 7538, "stylus_support": True, "price": 99000, "category": "professional"},
    {"brand": "Xiaomi", "model": "Pad 5 Pro", "display": "12.4 AMOLED QHD+", "processor": "Snapdragon 870", "ram": "8GB", "storage": "256GB", "battery": 10100, "stylus_support": True, "price": 52000, "category": "professional"},
    {"brand": "Samsung", "model": "Galaxy Tab S8 Ultra", "display": "14.6 AMOLED QHD 120Hz", "processor": "Snapdragon 8 Gen 1", "ram": "12GB", "storage": "256GB", "battery": 11200, "stylus_support": True, "price": 95000, "category": "professional"},
    {"brand": "Apple", "model": "iPad Pro 12.9 (6th Gen)", "display": "12.9 Liquid Retina XDR", "processor": "Apple M2", "ram": "8GB", "storage": "256GB", "battery": 10308, "stylus_support": True, "price": 129000, "category": "professional"},
    {"brand": "Lenovo", "model": "Tab P12 Pro", "display": "12.6 OLED QHD+", "processor": "Snapdragon 870", "ram": "8GB", "storage": "128GB", "battery": 10200, "stylus_support": True, "price": 65000, "category": "professional"},
    {"brand": "Xiaomi", "model": "Redmi Pad", "display": "10.95 IPS FHD+", "processor": "MediaTek Helio G99", "ram": "4GB", "storage": "128GB", "battery": 10000, "stylus_support": False, "price": 14000, "category": "entertainment"},
    {"brand": "Samsung", "model": "Galaxy Tab A8", "display": "10.5 IPS FHD", "processor": "MediaTek Helio G80", "ram": "4GB", "storage": "64GB", "battery": 7040, "stylus_support": False, "price": 18000, "category": "entertainment"},
    {"brand": "Apple", "model": "iPad Mini (6th Gen)", "display": "8.3 Liquid Retina", "processor": "Apple A15 Bionic", "ram": "4GB", "storage": "64GB", "battery": 5124, "stylus_support": True, "price": 39000, "category": "student"},
    {"brand": "OnePlus", "model": "Pad (256GB)", "display": "11.61 AMOLED QHD 144Hz", "processor": "MediaTek Dimensity 9000", "ram": "12GB", "storage": "256GB", "battery": 9510, "stylus_support": False, "price": 42000, "category": "entertainment"},
    {"brand": "Lenovo", "model": "Yoga Tab 11", "display": "11.5 OLED QHD", "processor": "MediaTek Kompanio 1300T", "ram": "8GB", "storage": "256GB", "battery": 13010, "stylus_support": True, "price": 58000, "category": "professional"},
    {"brand": "Samsung", "model": "Galaxy Tab S7 FE", "display": "12.4 IPS FHD", "processor": "Exynos 5080", "ram": "4GB", "storage": "64GB", "battery": 10090, "stylus_support": True, "price": 28000, "category": "student"},
    {"brand": "Apple", "model": "iPad (11th Gen)", "display": "10.9 Liquid Retina", "processor": "Apple A15 Bionic", "ram": "4GB", "storage": "256GB", "battery": 8022, "stylus_support": True, "price": 42000, "category": "student"},
]

HEADPHONES_DATA = [
    {"brand": "Sony", "model": "WH-CH720N", "type": "over-ear", "connectivity": "wireless", "anc": True, "battery": 35, "driver_size": 40, "price": 12000, "category": "budget"},
    {"brand": "JBL", "model": "Tune 750TNC", "type": "over-ear", "connectivity": "wireless", "anc": True, "battery": 30, "driver_size": 40, "price": 11000, "category": "budget"},
    {"brand": "Sennheiser", "model": "HD 450SE", "type": "over-ear", "connectivity": "wireless", "anc": True, "battery": 25, "driver_size": 32, "price": 9500, "category": "budget"},
    {"brand": "Apple", "model": "AirPods Pro", "type": "in-ear", "connectivity": "wireless", "anc": True, "battery": 6, "driver_size": 8.8, "price": 26000, "category": "audiophile"},
    {"brand": "Sony", "model": "WF-1000XM5", "type": "earbuds", "connectivity": "wireless", "anc": True, "battery": 12, "driver_size": 8.6, "price": 24000, "category": "audiophile"},
    {"brand": "Bose", "model": "QuietComfort 45", "type": "over-ear", "connectivity": "wireless", "anc": True, "battery": 24, "driver_size": 40, "price": 30000, "category": "audiophile"},
    {"brand": "Samsung", "model": "Galaxy Buds2 Pro", "type": "earbuds", "connectivity": "wireless", "anc": True, "battery": 8, "driver_size": 6.5, "price": 13000, "category": "gaming"},
    {"brand": "Beats", "model": "Fit Pro", "type": "earbuds", "connectivity": "wireless", "anc": True, "battery": 8, "driver_size": 5.5, "price": 21000, "category": "gaming"},
    {"brand": "Jabra", "model": "Elite Active 75t", "type": "earbuds", "connectivity": "wireless", "anc": True, "battery": 7.5, "driver_size": 6.8, "price": 16000, "category": "workout"},
    {"brand": "Sony", "model": "WH-XB910N", "type": "over-ear", "connectivity": "wireless", "anc": True, "battery": 24, "driver_size": 40, "price": 18000, "category": "gaming"},
    {"brand": "Skullcandy", "model": "Push Ultra", "type": "earbuds", "connectivity": "wireless", "anc": False, "battery": 7, "driver_size": 6, "price": 8000, "category": "budget"},
    {"brand": "Anker", "model": "Soundcore Space Q45", "type": "over-ear", "connectivity": "wireless", "anc": True, "battery": 50, "driver_size": 40, "price": 16000, "category": "budget"},
    {"brand": "JBL", "model": "Reflect Flow Pro", "type": "earbuds", "connectivity": "wireless", "anc": False, "battery": 10, "driver_size": 6.8, "price": 10500, "category": "workout"},
    {"brand": "Sennheiser", "model": "Momentum 4", "type": "over-ear", "connectivity": "wireless", "anc": True, "battery": 60, "driver_size": 42, "price": 28000, "category": "audiophile"},
    {"brand": "Apple", "model": "AirPods Max", "type": "over-ear", "connectivity": "wireless", "anc": True, "battery": 20, "driver_size": 40, "price": 55000, "category": "audiophile"},
]

SMARTWATCHES_DATA = [
    {"brand": "Apple", "model": "Watch Series 8", "display": "1.9 Retina LTPO", "battery": 1.5, "sensors": ["Heart Rate", "ECG", "SpO2", "Temp"], "water_resistance": "50m", "compatibility": "iOS", "price": 42000, "category": "luxury"},
    {"brand": "Samsung", "model": "Galaxy Watch5", "display": "1.4 AMOLED", "battery": 1.75, "sensors": ["Heart Rate", "ECG", "SpO2"], "water_resistance": "50m", "compatibility": "Android", "price": 32000, "category": "luxury"},
    {"brand": "Garmin", "model": "Forerunner 255", "display": "1.3 AMOLED", "battery": 11, "sensors": ["GPS", "Heart Rate", "Barometer"], "water_resistance": "50m", "compatibility": "Both", "price": 28000, "category": "fitness"},
    {"brand": "Fitbit", "model": "Sense 2", "display": "1.58 AMOLED", "battery": 6, "sensors": ["Heart Rate", "SpO2", "EDA", "Skin Temp"], "water_resistance": "50m", "compatibility": "Both", "price": 16000, "category": "fitness"},
    {"brand": "Amazfit", "model": "GTR 3 Pro", "display": "1.45 AMOLED", "battery": 12, "sensors": ["GPS", "Heart Rate", "SpO2"], "water_resistance": "100m", "compatibility": "Both", "price": 13000, "category": "fitness"},
    {"brand": "Apple", "model": "Watch Ultra", "display": "2.04 Retina LTPO", "battery": 2, "sensors": ["Heart Rate", "ECG", "SpO2", "Compass"], "water_resistance": "100m", "compatibility": "iOS", "price": 65000, "category": "luxury"},
    {"brand": "Garmin", "model": "Epix Gen 2", "display": "1.3 AMOLED", "battery": 10, "sensors": ["GPS", "Heart Rate", "Barometer"], "water_resistance": "100m", "compatibility": "Both", "price": 52000, "category": "fitness"},
    {"brand": "Samsung", "model": "Galaxy Watch6 Classic", "display": "1.5 AMOLED", "battery": 2.5, "sensors": ["Heart Rate", "ECG", "SpO2"], "water_resistance": "50m", "compatibility": "Android", "price": 35000, "category": "luxury"},
    {"brand": "Fitbit", "model": "Versa 4", "display": "1.58 AMOLED", "battery": 6, "sensors": ["Heart Rate", "SpO2"], "water_resistance": "50m", "compatibility": "Both", "price": 18000, "category": "fitness"},
    {"brand": "Amazfit", "model": "GTS 4", "display": "1.65 AMOLED", "battery": 14, "sensors": ["GPS", "Heart Rate", "SpO2"], "water_resistance": "50m", "compatibility": "Both", "price": 9000, "category": "casual"},
]

In [5]:

# ============================================================================
# FIELD VALIDATION RULES
# ============================================================================

REQUIRED_FIELDS = {
    "laptop": ["brand", "model", "price", "processor", "ram"],
    "phone": ["brand", "model", "price", "processor", "ram"],
    "tablet": ["brand", "model", "price", "processor", "ram"],
    "headphone": ["brand", "model", "price", "type"],
    "smartwatch": ["brand", "model", "price", "display"],
}


In [6]:
# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def validate_product(product: Dict[str, Any], product_type: str, index: int) -> bool:
    """Validate that product has all required fields."""
    required = REQUIRED_FIELDS.get(product_type, [])
    missing = [field for field in required if field not in product]
    
    if missing:
        print(f"⚠️  {product_type}_{index}: Missing fields: {missing}")
        return False
    
    # Validate price is numeric
    if not isinstance(product.get("price"), (int, float)):
        print(f"⚠️  {product_type}_{index}: Price must be numeric, got {type(product['price'])}")
        return False
    
    return True


def sanitize_metadata(product: Dict[str, Any]) -> Dict[str, Any]:
    """Convert lists and non-serializable fields to strings for ChromaDB."""
    sanitized = {}
    
    for key, value in product.items():
        if isinstance(value, list):
            # Convert lists to comma-separated strings
            sanitized[key] = ", ".join(str(v) for v in value)
        elif isinstance(value, bool):
            # Convert booleans to strings (ChromaDB compatible)
            sanitized[key] = "Yes" if value else "No"
        elif isinstance(value, (int, float, str)):
            # These are fine as-is
            sanitized[key] = value
        else:
            # Convert anything else to string
            sanitized[key] = str(value)
    
    return sanitized


def create_document_string(product: Dict[str, Any], product_type: str) -> str:
    """Create searchable document string based on product type."""
    
    if product_type == "laptop":
        return (
            f"{product['brand']} {product['model']} | "
            f"Processor: {product['processor']} | "
            f"RAM: {product['ram']} | "
            f"Storage: {product['storage']} | "
            f"Display: {product['display']} | "
            f"Graphics: {product.get('graphics', 'Integrated')} | "
            f"Price: ₹{product['price']} | "
            f"Category: {product['category']} | "
            f"Battery: {product.get('battery', 'N/A')}Wh | "
            f"Weight: {product.get('weight', 'N/A')}kg"
        )
    
    elif product_type == "phone":
        return (
            f"{product['brand']} {product['model']} | "
            f"Processor: {product['processor']} | "
            f"RAM: {product['ram']} | "
            f"Storage: {product['storage']} | "
            f"Camera: {product['camera']} | "
            f"Display: {product['display']} | "
            f"Battery: {product['battery']}mAh | "
            f"OS: {product.get('os', 'N/A')} | "
            f"Price: ₹{product['price']} | "
            f"Category: {product['category']}"
        )
    
    elif product_type == "tablet":
        stylus = "Stylus Supported" if product.get('stylus_support') else "No Stylus"
        return (
            f"{product['brand']} {product['model']} | "
            f"Display: {product['display']} | "
            f"Processor: {product['processor']} | "
            f"RAM: {product['ram']} | "
            f"Storage: {product['storage']} | "
            f"Battery: {product['battery']}mAh | "
            f"{stylus} | "
            f"Price: ₹{product['price']} | "
            f"Category: {product['category']}"
        )
    
    elif product_type == "headphone":
        anc_text = "ANC: Yes" if product.get('anc') else "ANC: No"
        return (
            f"{product['brand']} {product['model']} | "
            f"Type: {product['type']} | "
            f"Connectivity: {product['connectivity']} | "
            f"{anc_text} | "
            f"Battery: {product['battery']}h | "
            f"Driver: {product.get('driver_size', 'N/A')}mm | "
            f"Price: ₹{product['price']} | "
            f"Category: {product['category']}"
        )
    
    elif product_type == "smartwatch":
        sensors = product.get('sensors', [])
        sensors_text = ", ".join(sensors) if isinstance(sensors, list) else str(sensors)
        return (
            f"{product['brand']} {product['model']} | "
            f"Display: {product['display']} | "
            f"Battery: {product['battery']} days | "
            f"Sensors: {sensors_text} | "
            f"Water Resistance: {product.get('water_resistance', 'N/A')} | "
            f"Compatibility: {product.get('compatibility', 'N/A')} | "
            f"Price: ₹{product['price']} | "
            f"Category: {product['category']}"
        )
    
    return ""


def generate_product_hash(brand: str, model: str, product_type: str) -> str:
    """Generate unique hash to detect duplicates."""
    content = f"{product_type}_{brand}_{model}".lower()
    return hashlib.md5(content.encode()).hexdigest()[:8]


def save_json_files(base_dir: str = "database") -> Dict[str, str]:
    """Save product data to JSON files. Returns dict of file paths."""
    Path(base_dir).mkdir(parents=True, exist_ok=True)
    
    product_files = {
        "laptop": (LAPTOPS_DATA, f"{base_dir}/laptops.json"),
        "phone": (PHONES_DATA, f"{base_dir}/phones.json"),
        "tablet": (TABLETS_DATA, f"{base_dir}/tablets.json"),
        "headphone": (HEADPHONES_DATA, f"{base_dir}/headphones.json"),
        "smartwatch": (SMARTWATCHES_DATA, f"{base_dir}/smartwatches.json"),
    }
    
    saved_files = {}
    for product_type, (data, filepath) in product_files.items():
        with open(filepath, "w") as f:
            json.dump(data, f, indent=2)
        saved_files[product_type] = filepath
    
    return saved_files


def load_products_from_json(base_dir: str = "database") -> Dict[str, List[Dict[str, Any]]]:
    """Load all products from JSON files with error handling."""
    products = {}
    product_types = ["laptop", "phone", "tablet", "headphone", "smartwatch"]
    
    for product_type in product_types:
        filepath = f"{base_dir}/{product_type}s.json"
        try:
            with open(filepath, "r") as f:
                products[product_type] = json.load(f)
        except FileNotFoundError:
            print(f"⚠️  Warning: {filepath} not found, skipping {product_type}s")
            products[product_type] = []
        except json.JSONDecodeError as e:
            print(f"❌ Error: {filepath} has invalid JSON: {e}")
            products[product_type] = []
    
    return products

In [7]:

# ============================================================================
# MAIN BUILDER FUNCTION
# ============================================================================

def build_knowledge_base(base_dir: str = "database") -> None:
    """
    Build ChromaDB knowledge base with all products.
    
    Steps:
    1. Save inline data to JSON files
    2. Load and validate all products
    3. Generate embeddings with GPU if available
    4. Store in ChromaDB with cosine similarity
    """
    
    print("=" * 60)
    print("🔨 KNOWLEDGE BASE BUILDER")
    print("=" * 60)
    
    # Step 1: Save JSON files
    print("\n📁 Saving product data to JSON files...")
    saved_files = save_json_files(base_dir)
    for ptype, filepath in saved_files.items():
        print(f"   ✅ {filepath}")
    
    # Step 2: Load products
    print("\n📦 Loading products...")
    products_dict = load_products_from_json(base_dir)
    
    # Step 3: Validate and count
    print("\n🔍 Validating products...")
    product_counts = {}
    total_valid = 0
    total_invalid = 0
    duplicate_hashes = set()
    duplicates_found = 0
    
    validated_products = {}
    for product_type, products in products_dict.items():
        valid_list = []
        for idx, product in enumerate(products):
            # Validate required fields
            if not validate_product(product, product_type, idx):
                total_invalid += 1
                continue
            
            # Check for duplicates
            product_hash = generate_product_hash(
                product["brand"], product["model"], product_type
            )
            if product_hash in duplicate_hashes:
                print(f"   ⚠️  Duplicate found: {product['brand']} {product['model']} ({product_type})")
                duplicates_found += 1
                continue
            
            duplicate_hashes.add(product_hash)
            valid_list.append(product)
            total_valid += 1
        
        validated_products[product_type] = valid_list
        product_counts[product_type] = len(valid_list)
    
    # Step 4: Summary
    print(f"\n📊 Products loaded:")
    for ptype, count in product_counts.items():
        print(f"   • {count} {ptype}s")
    print(f"   ✅ Total valid: {total_valid}")
    if total_invalid > 0:
        print(f"   ⚠️  Skipped invalid: {total_invalid}")
    if duplicates_found > 0:
        print(f"   ⚠️  Duplicates removed: {duplicates_found}")
    
    if total_valid == 0:
        print("\n❌ No valid products found. Aborting.")
        sys.exit(1)
    
    # Step 5: Initialize ChromaDB
    print(f"\n🗄️  Initializing ChromaDB at: {base_dir}/chroma_db")
    chroma_db_path = f"{base_dir}/chroma_db"
    client = chromadb.PersistentClient(path=chroma_db_path)
    
    # Delete existing collection for fresh start
    try:
        client.delete_collection(name=COLLECTION_NAME)
        print(f"   🗑️  Deleted existing '{COLLECTION_NAME}' collection")
    except Exception:
        pass
    
    # Create new collection
    try:
        collection = client.get_or_create_collection(
            name=COLLECTION_NAME,
            metadata={
                "hnsw:space": SIMILARITY_METRIC,
                "description": "Multi-category product database with embeddings"
            }
        )
        print(f"   ✅ Created new '{COLLECTION_NAME}' collection")
    except Exception as e:
        print(f"   ❌ Failed to create collection: {e}")
        sys.exit(1)
    
    # Step 6: Load embedding model
    print(f"\n🧠 Loading embedding model: {MODEL_NAME}")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    if device == "cuda":
        print(f"   🚀 GPU detected: {torch.cuda.get_device_name(0)}")
        print(f"   💾 VRAM available: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
    else:
        print(f"   ⚠️  No GPU found, using CPU (slower)")
    
    try:
        embedding_model = SentenceTransformer(MODEL_NAME, device=device)
        print(f"   ✅ Model loaded on {device.upper()}")
    except Exception as e:
        print(f"   ❌ Failed to load model: {e}")
        sys.exit(1)
    
    # Step 7: Generate embeddings
    print(f"\n⚡ Generating embeddings for {total_valid} products...")
    
    all_ids = []
    all_documents = []
    all_embeddings = []
    all_metadatas = []
    
    for product_type, products in validated_products.items():
        desc = f"🔹 {product_type.capitalize()}s"
        for idx, product in enumerate(tqdm(products, desc=desc, unit="prod")):
            # Create unique ID
            product_id = f"{product_type}_{idx}"
            
            # Create searchable document string
            document = create_document_string(product, product_type)
            
            # Generate embedding (normalized for cosine similarity)
            embedding = embedding_model.encode(
                document, 
                normalize_embeddings=True,
                show_progress_bar=False
            )
            
            # Sanitize metadata (convert lists/booleans to strings)
            metadata = sanitize_metadata(product)
            metadata["product_type"] = product_type
            metadata["document_text"] = document  # Store original text for retrieval
            
            all_ids.append(product_id)
            all_documents.append(document)
            all_embeddings.append(embedding.tolist())
            all_metadatas.append(metadata)
    
    # Step 8: Batch insert into ChromaDB
    print(f"\n💾 Storing {len(all_ids)} embeddings in ChromaDB...")
    
    for i in tqdm(range(0, len(all_ids), BATCH_SIZE), desc="Inserting batches", unit="batch"):
        batch_end = min(i + BATCH_SIZE, len(all_ids))
        
        try:
            collection.add(
                ids=all_ids[i:batch_end],
                embeddings=all_embeddings[i:batch_end],
                documents=all_documents[i:batch_end],
                metadatas=all_metadatas[i:batch_end]
            )
        except Exception as e:
            print(f"\n   ❌ Error inserting batch {i//BATCH_SIZE}: {e}")
            print(f"   Continuing with remaining products...")
            continue
    
    # Step 9: Final summary
    final_count = collection.count()
    print(f"\n{'=' * 60}")
    print(f"✅ DATABASE BUILD COMPLETE")
    print(f"{'=' * 60}")
    print(f"📁 Database path: {chroma_db_path}")
    print(f"📦 Collection: {COLLECTION_NAME}")
    print(f"🔢 Products stored: {final_count}")
    print(f"📏 Similarity metric: {SIMILARITY_METRIC}")
    print(f"🖥️  Device used: {device.upper()}")
    print(f"\n📋 Product breakdown:")
    for ptype, count in product_counts.items():
        print(f"   • {count} {ptype}s")
    print(f"\n💡 Tip: Run 'python retriever.py' to test search queries.")
    print(f"{'=' * 60}\n")



In [8]:
# ============================================================================
# MAIN ENTRY POINT
# ============================================================================

if __name__ == "__main__":
    try:
        build_knowledge_base()
    except KeyboardInterrupt:
        print("\n\n⚠️  Build cancelled by user.")
        sys.exit(0)
    except Exception as e:
        print(f"\n❌ Unexpected error: {e}")
        import traceback
        traceback.print_exc()
        sys.exit(1)

🔨 KNOWLEDGE BASE BUILDER

📁 Saving product data to JSON files...
   ✅ database/laptops.json
   ✅ database/phones.json
   ✅ database/tablets.json
   ✅ database/headphones.json
   ✅ database/smartwatches.json

📦 Loading products...
⚠️  Warning: database/smartwatchs.json not found, skipping smartwatchs

🔍 Validating products...

📊 Products loaded:
   • 30 laptops
   • 25 phones
   • 20 tablets
   • 15 headphones
   • 0 smartwatchs
   ✅ Total valid: 90

🗄️  Initializing ChromaDB at: database/chroma_db
   ✅ Created new 'products' collection

🧠 Loading embedding model: all-MiniLM-L6-v2
   ⚠️  No GPU found, using CPU (slower)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4798.76it/s]


   ✅ Model loaded on CPU

⚡ Generating embeddings for 90 products...


🔹 Headphones: 100%|██████████| 15/15 [00:00<00:00, 74.86prod/s]
🔹 Smartwatchs: 0prod [00:00, ?prod/s]



💾 Storing 90 embeddings in ChromaDB...


Inserting batches: 100%|██████████| 1/1 [00:00<00:00, 10.58batch/s]


✅ DATABASE BUILD COMPLETE
📁 Database path: database/chroma_db
📦 Collection: products
🔢 Products stored: 90
📏 Similarity metric: cosine
🖥️  Device used: CPU

📋 Product breakdown:
   • 30 laptops
   • 25 phones
   • 20 tablets
   • 15 headphones
   • 0 smartwatchs

💡 Tip: Run 'python retriever.py' to test search queries.

